# 02 - Feature Extraction & Industrial Anomaly Detection Pipeline
**Author:** Tanya Tejaswi (Computer Vision & Anomaly Detection)

### Complete Workflow:
1. Preprocess images & extract multi-scale spatial patch features.
2. Export global image embeddings (`image_embeddings.npy`) and metadata for Gayathri's PCA, t-SNE, and clustering pipeline.
3. Fit **PatchCore Memory Bank** anomaly detectors strictly on pristine normal units (`train/good`).
4. Evaluate continuous anomaly scoring, ROC-AUC, PR-AUC, F1-scores, and calibrated decision thresholds across all **8 Industrial Categories**:
   - `pcb1`, `pcb2`, `pcb3`, `pcb4`, `capsules`, `bottle`, `cable`, `steel`.
5. Serialize production models to `models/memory_bank_{category}.pkl`.

In [ ]:
import os
import sys
import tempfile
os.environ["MPLCONFIGDIR"] = tempfile.gettempdir()
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

sys.path.append("..")
from src.anomaly_detection import MemoryBankAnomalyDetector, AnomalyInferenceEngine
from src.evaluation import compute_anomaly_detection_metrics, evaluate_image_level, plot_roc_pr_curves
from src.visualization import plot_anomaly_comparison

print("✅ Anomaly detection modules loaded!")

### 1. Ingest Master Embeddings & Verify Deliverables

In [ ]:
emb_path = Path("../data/embeddings/image_embeddings.npy")
meta_path = Path("../data/embeddings/metadata.csv")

assert emb_path.exists() and meta_path.exists(), "Embeddings deliverable missing!"
features = np.load(emb_path)
meta_df = pd.read_csv(meta_path)

print(f"🎉 Deliverables verified for Gayathri's clustering pipeline:")
print(f"  - Embeddings Array: {features.shape} ({emb_path.stat().st_size / 1e6:.2f} MB)")
print(f"  - Metadata Index:   {len(meta_df)} rows across {meta_df['category'].nunique()} categories")

### 2. Multi-Category Memory Bank Inspection & Benchmark Scoreboard
Load the trained production memory banks (`models/memory_bank_{category}.pkl`) and verify the benchmark scoreboard.

In [ ]:
summary_csv = Path("../reports/anomaly_detection_summary.csv")
if summary_csv.exists():
    summary_df = pd.read_csv(summary_csv)
    print("🏆 8-Line Industrial Anomaly Detection Performance Scoreboard:")
    print(summary_df.to_string(index=False))
else:
    print("Run scripts/train_industrial_suite.py to generate summary scoreboard.")

### 3. Model Smoke Test: AnomalyInferenceEngine Deployment

In [ ]:
# Smoke test the inference engine on bottle and pcb1
engine = AnomalyInferenceEngine(category="bottle", models_dir="../models")
test_img = Path("../data/mvtec/bottle/test/broken_large/000.png")

if test_img.exists():
    res = engine.predict(test_img)
    print("✅ Live AnomalyInferenceEngine Prediction Succeeded:")
    print(f"  - Category:    {res['category']}")
    print(f"  - Score:       {res['score']:.4f}")
    print(f"  - Threshold:   {res['threshold']:.4f}")
    print(f"  - Is Defect:   {res['is_defect']}")
    print(f"  - Anomaly Map: {res['anomaly_map'].shape}")
    
    # Display inspection overlay
    plt.figure(figsize=(4, 4), dpi=120)
    plt.imshow(res['heatmap_overlay'])
    plt.title(f"Thermal Heatmap Overlay (Score={res['score']:.1f})", fontsize=10, fontweight='bold')
    plt.axis('off')
    plt.show()